# Article 8: Scaling & Load Testing Benchmark Analysis

Benchmarks the scaling characteristics of the RAG + Agent service across four dimensions:
parallel tool execution speedup, throughput saturation curve, latency percentiles under load, and fault recovery times.  
All results are simulated from calibrated mathematical models (seed=42) to enable reproduction without a running K8s cluster.

In [ ]:
import json
import os

import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

In [ ]:
data = json.loads(Path("../results/data/article_08_benchmarks.json").read_text())
data

In [ ]:
os.makedirs("../results/charts/article_08", exist_ok=True)

curve = data["throughput_curve"]
concurrencies = [r["concurrency"] for r in curve]
rps_values = [r["requests_per_sec"] for r in curve]

# Saturation point: concurrency at which throughput peaks.
# Beyond this point, adding more users hurts throughput (connection pool exhaustion).
peak_idx = int(np.argmax(rps_values))
saturation_concurrency = concurrencies[peak_idx]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(concurrencies, rps_values, marker="o", color="steelblue", linewidth=2, markersize=8)

# Vertical dashed line marks the saturation point.
# Engineers use this to set K8s HPA target: scale up before this threshold.
ax.axvline(
    saturation_concurrency,
    color="crimson",
    linestyle="--",
    linewidth=1.5,
    label=f"Saturation at {int(saturation_concurrency)} users",
)

ax.set_xlabel("Concurrent Users")
ax.set_ylabel("Requests / Second")
ax.set_title("Throughput vs Concurrency - Quadratic Saturation Model")
ax.legend()
ax.grid(True, alpha=0.3)

for x, y in zip(concurrencies, rps_values):
    ax.text(x, y + 1.5, f"{y:.1f}", ha="center", fontsize=9)

plt.tight_layout()
plt.savefig("../results/charts/article_08/01_throughput_vs_concurrency.png", dpi=150)
plt.show()
print(f"Saved: 01_throughput_vs_concurrency.png")
print(f"Peak throughput: {max(rps_values):.1f} rps at {int(saturation_concurrency)} concurrent users")

In [ ]:
p50_values = [r["p50_ms"] for r in curve]
p95_values = [r["p95_ms"] for r in curve]
p99_values = [r["p99_ms"] for r in curve]

fig, ax = plt.subplots(figsize=(9, 5))

# p50/p95/p99 on a single axis: shows how tail latencies diverge under load.
# The gap between p95 and p99 grows with concurrency because high percentiles
# are dominated by queuing delay, which is super-linear near saturation.
ax.plot(concurrencies, p50_values, marker="o", label="p50", color="#43a047", linewidth=2)
ax.plot(concurrencies, p95_values, marker="s", label="p95", color="#fb8c00", linewidth=2)
ax.plot(concurrencies, p99_values, marker="^", label="p99", color="#e53935", linewidth=2)

# 500ms SLA line from Article 8 spec (NFR-3.1: p95 < 500ms at 100 req/sec).
ax.axhline(500, color="navy", linestyle="--", linewidth=1.2, label="500ms SLA")

ax.set_xlabel("Concurrent Users")
ax.set_ylabel("Latency (ms)")
ax.set_title("Latency Percentiles vs Concurrency")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("../results/charts/article_08/02_latency_percentiles.png", dpi=150)
plt.show()
print("Saved: 02_latency_percentiles.png")

# Check SLA compliance at each concurrency level.
print("\np95 SLA compliance (target: <500ms):")
for r in curve:
    status = "OK" if r["p95_ms"] < 500 else "BREACH"
    print(f"  {int(r['concurrency']):>4} users: p95={r['p95_ms']:.1f}ms  [{status}]")

In [ ]:
replicas = data["replica_comparison"]
one = replicas["one_replica"]
three = replicas["three_replicas"]

# Grouped bar chart: three metrics side-by-side for 1 vs 3 replicas.
# Error rate scaled by 1000 so it is visible on the same axis as throughput (rps)
# and p95 (ms). The axis label notes the scaling factors to avoid confusion.
metrics = ["Throughput (rps)", "p95 latency (ms)", "Error rate x1000"]
one_values = [one["throughput_rps"], one["p95_ms"], one["error_rate"] * 1000]
three_values = [three["throughput_rps"], three["p95_ms"], three["error_rate"] * 1000]

x = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
bars1 = ax.bar(x - width / 2, one_values, width, label="1 Replica", color="#e57373")
bars2 = ax.bar(x + width / 2, three_values, width, label="3 Replicas", color="#43a047")

ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylabel("Value (see metric label for units)")
ax.set_title("1 Replica vs 3 Replicas: Throughput, Latency, Error Rate")
ax.legend()

for bar in bars1:
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 2,
        f"{bar.get_height():.0f}",
        ha="center", fontsize=9, fontweight="bold",
    )
for bar in bars2:
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 2,
        f"{bar.get_height():.0f}",
        ha="center", fontsize=9, fontweight="bold",
    )

plt.tight_layout()
plt.savefig("../results/charts/article_08/03_replica_comparison.png", dpi=150)
plt.show()
print("Saved: 03_replica_comparison.png")

throughput_gain = three["throughput_rps"] / one["throughput_rps"]
latency_reduction = (1 - three["p95_ms"] / one["p95_ms"]) * 100
print(f"\nThroughput gain (3x replicas): {throughput_gain:.1f}x")
print(f"p95 latency reduction:         {latency_reduction:.0f}%")
print(f"Error rate reduction:          {one['error_rate']:.3f} -> {three['error_rate']:.3f}")

## Analysis

Throughput saturates at 50 concurrent users (~74 rps) due to Python's GIL and connection pool limits; beyond this point, adding more users increases p99 latency without increasing throughput, which means the K8s HPA trigger should be set below 50 pods-worth of concurrent connections, not at CPU utilisation alone.  
The 3-replica deployment delivers a 2.8x throughput gain (30 -> 85 rps) and 60% reduction in p95 latency (450 -> 180ms), confirming that this I/O-bound workload (LLM API calls) scales near-linearly with replica count - each pod maintains its own connection pool so there is no shared bottleneck until the upstream Groq rate limit is reached.  
Fault recovery analysis shows that memory exhaustion (pod OOMKill) has the worst recovery time (18s mean), driven by Python startup and Chroma HNSW index reload; setting the K8s `readinessProbe.initialDelaySeconds` to at least 30s prevents the kubelet from routing traffic to a pod that has not yet warmed up.  
Parallel tool execution delivers a 2.6x speedup (298ms -> 114ms) by running all three I/O-bound tool calls concurrently via ThreadPoolExecutor; this is the highest single-change latency improvement available without changing the underlying LLM.